In [ ]:
from google.colab import drive
drive.mount("/content/MyDrive")

Mounted at /content/MyDrive


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

dataset_path = '/content/final_data/final_data'

# Training generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

# Validation generator (only rescale)
val_datagen = ImageDataGenerator(rescale=1./255)

# Create generators
train_generator = train_datagen.flow_from_directory(
    os.path.join(dataset_path, 'train'),
    target_size=(28,28),
    color_mode='grayscale',
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    os.path.join(dataset_path, 'val'),
    target_size=(28,28),
    color_mode='grayscale',
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)


Found 10500 images belonging to 3 classes.
Found 2250 images belonging to 3 classes.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import os

# ---------- Paths ----------
dataset_path = '/content/final_data/final_data'

# ---------- Data generators ----------
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(dataset_path, 'train'),
    target_size=(32,32),
    color_mode='grayscale',
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    os.path.join(dataset_path, 'val'),
    target_size=(32,32),
    color_mode='grayscale',
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# ---------- Model ----------
base_model = EfficientNetB0(
    weights=None,
    include_top=False,
    input_shape=(32,32,1)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(3, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# ---------- Callbacks ----------
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "/content/best_model.h5",
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2,
    verbose=1
)

# ---------- Training ----------
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=[early_stop, checkpoint, reduce_lr]
)

# ---------- Save to Drive ----------
!cp /content/best_model.h5 /content/drive/MyDrive/
print("Best model saved to MyDrive")


Found 10500 images belonging to 3 classes.
Found 2250 images belonging to 3 classes.
Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


327/329 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - accuracy: 0.4144 - loss: 1.2280
Epoch 1: val_accuracy improved from -inf to 0.33333, saving model to /content/best_model.h5


329/329 ━━━━━━━━━━━━━━━━━━━━ 142s 200ms/step - accuracy: 0.4149 - loss: 1.2266 - val_accuracy: 0.3333 - val_loss: 1.1668 - learning_rate: 0.0010
Epoch 2/50
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6186 - loss: 0.8710
Epoch 2: val_accuracy improved from 0.33333 to 0.49378, saving model to /content/best_model.h5


329/329 ━━━━━━━━━━━━━━━━━━━━ 12s 36ms/step - accuracy: 0.6187 - loss: 0.8708 - val_accuracy: 0.4938 - val_loss: 1.2200 - learning_rate: 0.0010
Epoch 3/50
328/329 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.7347 - loss: 0.6673
Epoch 3: val_accuracy improved from 0.49378 to 0.70444, saving model to /content/best_model.h5


...

329/329 ━━━━━━━━━━━━━━━━━━━━ 12s 38ms/step - accuracy: 0.9530 - loss: 0.1404 - val_accuracy: 0.9742 - val_loss: 0.0767 - learning_rate: 3.0000e-04
Epoch 18/50
328/329 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9484 - loss: 0.1403
Epoch 18: val_accuracy did not improve from 0.97422
329/329 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.9485 - loss: 0.1402 - val_accuracy: 0.9720 - val_loss: 0.0799 - learning_rate: 3.0000e-04
Epoch 19/50
327/329 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9522 - loss: 0.1381
Epoch 19: val_accuracy improved from 0.97422 to 0.97511, saving model to /content/best_model.h5


329/329 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.9522 - loss: 0.1380 - val_accuracy: 0.9751 - val_loss: 0.0749 - learning_rate: 3.0000e-04
Epoch 20/50
328/329 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9555 - loss: 0.1302
Epoch 20: val_accuracy did not improve from 0.97511
329/329 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.9555 - loss: 0.1302 - val_accuracy: 0.9742 - val_loss: 0.0786 - learning_rate: 3.0000e-04
Epoch 21/50
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9515 - loss: 0.1398
Epoch 21: val_accuracy did not improve from 0.97511

Epoch 21: ReduceLROnPlateau reducing learning rate to 9.000000427477062e-05.
329/329 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - accuracy: 0.9515 - loss: 0.1398 - val_accuracy: 0.9720 - val_loss: 0.0857 - learning_rate: 3.0000e-04
Epoch 22/50
328/329 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9589 - loss: 0.1169
Epoch 22: val_accuracy did not improve from 0.97511
329/329 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - accuracy: 0.9589

329/329 ━━━━━━━━━━━━━━━━━━━━ 12s 36ms/step - accuracy: 0.9558 - loss: 0.1231 - val_accuracy: 0.9764 - val_loss: 0.0678 - learning_rate: 9.0000e-05
Epoch 24/50
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9600 - loss: 0.1121
Epoch 24: val_accuracy improved from 0.97644 to 0.97778, saving model to /content/best_model.h5


329/329 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - accuracy: 0.9600 - loss: 0.1121 - val_accuracy: 0.9778 - val_loss: 0.0692 - learning_rate: 9.0000e-05
Epoch 25/50
328/329 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9634 - loss: 0.1008
Epoch 25: val_accuracy did not improve from 0.97778
329/329 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - accuracy: 0.9634 - loss: 0.1008 - val_accuracy: 0.9756 - val_loss: 0.0661 - learning_rate: 9.0000e-05
Epoch 26/50
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9570 - loss: 0.1223
Epoch 26: val_accuracy did not improve from 0.97778
329/329 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - accuracy: 0.9570 - loss: 0.1223 - val_accuracy: 0.9778 - val_loss: 0.0708 - learning_rate: 9.0000e-05
Epoch 27/50
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9592 - loss: 0.1152
Epoch 27: val_accuracy did not improve from 0.97778

Epoch 27: ReduceLROnPlateau reducing learning rate to 2.700000040931627e-05.
329/329 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - accuracy: 0.9592

329/329 ━━━━━━━━━━━━━━━━━━━━ 12s 36ms/step - accuracy: 0.9611 - loss: 0.1112 - val_accuracy: 0.9796 - val_loss: 0.0655 - learning_rate: 2.7000e-05
Epoch 29/50
327/329 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9631 - loss: 0.1032
Epoch 29: val_accuracy did not improve from 0.97956
329/329 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - accuracy: 0.9631 - loss: 0.1032 - val_accuracy: 0.9778 - val_loss: 0.0666 - learning_rate: 2.7000e-05
Epoch 30/50
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9593 - loss: 0.1102
Epoch 30: val_accuracy did not improve from 0.97956

Epoch 30: ReduceLROnPlateau reducing learning rate to 8.100000013655517e-06.
329/329 ━━━━━━━━━━━━━━━━━━━━ 11s 32ms/step - accuracy: 0.9593 - loss: 0.1102 - val_accuracy: 0.9778 - val_loss: 0.0678 - learning_rate: 2.7000e-05
Epoch 31/50
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9596 - loss: 0.1134
Epoch 31: val_accuracy did not improve from 0.97956
329/329 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.9596